# Sequence copying: behavioral baseline

Measure whether prior exposure to a token sequence improves next-token prediction in the pretrained `attn-only-2l` transformer. This experiment reproduces a standard repeated-sequence evaluation; it does not establish a specific circuit.

## Metric

Prediction loss is the negative log probability of the target token, measured in nats. We average over scored positions within each sequence, then over sequences. The primary effect is the paired difference between control and repeated-context loss on identical second-block targets. Positive values indicate a copying benefit.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from copy_lab.experiment import run  # noqa: E402

## Experimental design

Compare `BOS A A` with `BOS B A`, where A and B are independently sampled, uniformly distributed sequences of nonspecial token IDs. Accidental token matches are allowed. Both conditions predict the same second-block targets at the same absolute positions; the preceding block differs. Exclude the first token of each block because it has no within-block prefix available for matching.

Configuration: 32 tokens per block, 16 sequence pairs, seed 42, CPU execution. The initial run downloads pretrained weights and the tokenizer from Hugging Face. Evaluation processes one sequence at a time to bound memory use.

In [ ]:
summary = run(length=32, samples=16, seed=42)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(root / "results/copying.png")))
print("Copying benefit:", summary["paired_copying_benefit_nats"], "nats")

## Interpretation and limitations

A positive control-minus-repeated loss difference supports a behavioral copying benefit under this input distribution. A negative difference indicates worse prediction with repeated context. Neither result alone identifies the responsible attention heads.

The saved seed-42 run reports mean second-block loss of 3.33 nats with repeated context and 14.67 nats with unrelated context, a paired difference of 11.34 nats. The standard error across 16 sequence pairs is 0.19 nats. This uncertainty estimate does not capture variation across seeds, models, or training runs.

Random token inputs differ from natural language. Further experiments should test sensitivity to sequence length, random seed, and distractors before drawing broader conclusions. Attention inspection and head interventions remain unimplemented.

Rerunning overwrites the files in `results/`; preserve each run separately when comparing configurations.

## References

- [TransformerLens main demo](https://transformerlensorg.github.io/TransformerLens/generated/demos/Main_Demo.html): pretrained models and repeated-sequence induction analysis.
- [ARENA transformer interpretability](https://learn.arena.education/chapter1_transformer_interp/02_intro_mech_interp/): induction circuits and experimental methods.